# 04 - Recipe recommender: demo pantries + hide-ingredients evaluation
Needs the tables from notebook 01 (it rebuilds them from INDB). Results are saved to `MyDrive/AaharAI/results/`.
Do not save this notebook back to GitHub.

In [ ]:
GITHUB_USER = "OnYourLeftover"

from google.colab import drive
drive.mount("/content/drive")
!pip -q install rapidfuzz
import sys
!git -C /content/AaharAI pull || git clone https://github.com/{GITHUB_USER}/AaharAI.git /content/AaharAI
sys.path.insert(0, "/content/AaharAI")
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import pandas as pd
from src import indb
from src.recommender import Recommender, hide_test

RESULTS = Path("/content/drive/MyDrive/AaharAI/results"); RESULTS.mkdir(parents=True, exist_ok=True)
rec, kept, nutrition = indb.build_tables()
R = Recommender(rec, kept, nutrition)
print(len(R.core), "recipes indexed |", int(sum(1 for v in R.core.values() if len(v) >= 5)), "have >= 5 core ingredients")

## Manual pantry scenarios\nFor your report: for each scenario write down whether the top suggestions are sensible (aim for 10-15 scenarios; edit or add your own).

In [ ]:
SCENARIOS = [
 ["potato", "onion", "tomato", "green chilli"],
 ["spinach", "paneer", "onion", "tomato", "garlic", "ginger"],
 ["egg", "onion", "tomato", "green chilli"],
 ["cauliflower", "potato", "peas", "onion", "tomato"],
 ["chicken", "onion", "tomato", "ginger", "garlic", "yogurt"],
 ["bread", "egg", "milk", "butter"],
 ["rice", "peas", "carrot", "onion"],
 ["cabbage", "carrot", "capsicum", "onion", "spring onions"],
 ["banana", "milk", "yogurt"],
 ["potato", "cheese", "bread", "tomato", "capsicum"],
 ["cucumber", "yogurt", "onion", "tomato", "mint leaves"],
 ["chickpeas", "onion", "tomato", "ginger", "garlic"],
]
for pantry in SCENARIOS:
    print("\nPANTRY:", ", ".join(pantry))
    for h in R.recommend(pantry, k=4):
        n = R.nutrition.loc[h.recipe_code]
        flag = " [check]" if n.nutrition_flag == "check" else ""
        print(f"  {h.score:.2f} {h.name[:46]:46} have {len(h.matched)} missing {h.missing[:3]} | {n.kcal:.0f} kcal/{n.serving_unit}{flag}")

## Hide-ingredients test (fixed seed)\nHide 20/30/50% of a recipe's ingredients, optionally add unrelated pantry items, and check whether the source recipe is in the top 5. `extra` = number of unrelated items added to the pantry.

In [ ]:
res = pd.concat([hide_test(R, n_extra=e) for e in (0, 5, 10)])
res.to_csv(RESULTS / "recommender_hide_test.csv", index=False)
for metric in ("hit@5", "hit@1"):
    print(metric); print(res.pivot_table(index=["extra", "hidden"], columns="mode", values=metric).round(2).to_string(), "\n")